**© Copyright AIDENTIFY. All rights reserved.**

# Part 4 | Session 32: RAG 평가 (RAGAS) & LLM-as-a-Judge

## 📋 학습 목표

1️⃣ RAG 시스템의 성능을 평가하는 핵심 지표를 이해한다  
2️⃣ RAGAS 프레임워크를 사용하여 자동 평가를 수행한다  
3️⃣ Faithfulness, Relevancy 등 세부 평가 지표를 이해한다  
4️⃣ LLM-as-a-Judge 방식으로 응답 품질을 평가한다  

---

### 🖥️ 실습 환경
- **GPU**: 불필요 (API 기반)
- **필수 패키지**: ragas, langchain, openai 또는 ollama
- **LLM**: OpenAI API 또는 Ollama 로컬 모델

## 1️⃣ RAG 평가가 중요한 이유

RAG 시스템을 구축한 후, **실제로 잘 작동하는지** 평가하는 것이 핵심입니다.

### 🔑 RAG 시스템에서 발생할 수 있는 문제
- 🔹 **검색 실패**: 관련 문서를 찾지 못함
- 🔹 **컨텍스트 부족**: 검색은 했지만 핵심 정보가 누락
- 🔹 **환각(Hallucination)**: 검색 결과에 없는 내용을 생성
- 🔹 **불충분한 답변**: 질문에 대한 답이 불완전

### 📊 RAG 평가 지표 체계

```
RAG 평가
├── 검색 품질 (Retrieval)
│   ├── Context Precision (정밀도)
│   └── Context Recall (재현율)
└── 생성 품질 (Generation)
    ├── Faithfulness (충실도)
    └── Answer Relevancy (답변 관련성)
```

In [ ]:
# 📦 필수 패키지 확인
import importlib
import os

packages = ["ragas", "langchain", "langchain_openai", "langchain_community", "chromadb"]

print("📦 패키지 버전 확인")
print("=" * 40)
for pkg_name in packages:
    try:
        pkg = importlib.import_module(pkg_name)
        version = getattr(pkg, "__version__", "installed")
        print(f"  ✅ {pkg_name}: {version}")
    except ImportError:
        print(f"  ❌ {pkg_name}: 설치 필요")
        print(f"     pip install {pkg_name}")

In [ ]:
# 🔑 환경 설정
from dotenv import load_dotenv
load_dotenv()  # .env 파일에서 OPENAI_API_KEY 로드

api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print(f"✅ OpenAI API 키: 설정됨")
    USE_OLLAMA = False
else:
    print("⚠️ OpenAI API 키가 없습니다. Ollama를 사용합니다.")
    print("   .env 파일에 OPENAI_API_KEY를 설정하면 OpenAI를 사용합니다.")
    USE_OLLAMA = True


## 2️⃣ 평가용 RAG 시스템 구축

평가를 위해 간단한 RAG 시스템을 먼저 구축합니다.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.schema import Document

# 📄 평가용 문서 준비
eval_documents = [
    Document(
        page_content="""트랜스포머(Transformer)는 2017년 구글이 발표한 'Attention is All You Need' 논문에서 소개되었습니다.
셀프 어텐션 메커니즘을 핵심으로 사용하며, 기존 RNN/LSTM 대비 병렬 처리가 가능합니다.
인코더-디코더 구조로 이루어져 있으며, BERT는 인코더만, GPT는 디코더만 사용합니다.
멀티헤드 어텐션으로 다양한 관점에서 입력을 분석합니다.""",
        metadata={"source": "transformer.txt"}
    ),
    Document(
        page_content="""LoRA(Low-Rank Adaptation)는 2021년 마이크로소프트에서 발표한 효율적 파인튜닝 방법입니다.
전체 모델 파라미터를 동결하고, 작은 저랭크 행렬만 학습합니다.
메모리 사용량을 크게 줄이면서도 Full Fine-Tuning에 가까운 성능을 달성합니다.
랭크(r)가 높을수록 표현력이 좋지만 메모리도 더 사용합니다.
QLoRA는 4bit 양자화와 LoRA를 결합하여 소비자급 GPU에서도 대형 모델 학습이 가능합니다.""",
        metadata={"source": "lora.txt"}
    ),
    Document(
        page_content="""RAG(Retrieval-Augmented Generation)는 검색 증강 생성 기술입니다.
LLM의 환각(hallucination) 문제를 해결하기 위해 외부 지식을 활용합니다.
벡터 데이터베이스에 문서를 저장하고, 질문과 유사한 문서를 검색하여 LLM에 제공합니다.
ChromaDB, FAISS, Pinecone 등의 벡터 DB를 사용할 수 있습니다.
청킹 전략과 임베딩 모델 선택이 RAG 성능에 큰 영향을 미칩니다.""",
        metadata={"source": "rag.txt"}
    ),
    Document(
        page_content="""RLHF(Reinforcement Learning from Human Feedback)는 인간 피드백 강화학습입니다.
ChatGPT의 성공 뒤에는 RLHF 기술이 있습니다.
보상 모델(Reward Model)을 학습하고, PPO 알고리즘으로 정책을 최적화합니다.
DPO(Direct Preference Optimization)는 RLHF를 단순화한 방법으로, 보상 모델 없이 직접 최적화합니다.
GRPO는 그룹 상대 정책 최적화로 DeepSeek에서 제안한 방법입니다.""",
        metadata={"source": "rlhf.txt"}
    ),
]

# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30)
splits = text_splitter.split_documents(eval_documents)

# 벡터 스토어 생성
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(splits, embeddings, collection_name="eval_demo")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"✅ 평가용 RAG 시스템 구축 완료")
print(f"  📄 원본 문서: {len(eval_documents)}개")
print(f"  📊 분할 청크: {len(splits)}개")

In [ ]:
# 🤖 RAG 체인 구성
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

if USE_OLLAMA:
    from langchain_community.llms import Ollama
    llm = Ollama(model="qwen2.5:1.5b", temperature=0.3, num_ctx=4096)
else:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

rag_prompt = PromptTemplate(
    template="""다음 컨텍스트를 참고하여 질문에 답변해주세요.
컨텍스트에 없는 정보는 답하지 마세요.

컨텍스트:
{context}

질문: {question}
답변:""",
    input_variables=["context", "question"]
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": rag_prompt},
)

print(f"✅ RAG 체인 준비 완료 (LLM: {'Ollama qwen2.5:1.5b' if USE_OLLAMA else 'OpenAI gpt-4o-mini'})")


## 3️⃣ 평가 데이터셋 준비

RAG 평가를 위해 **질문, 정답, 검색 컨텍스트** 세트를 준비합니다.

In [ ]:
# 📋 평가 데이터셋 생성
eval_questions = [
    "트랜스포머 모델은 언제 발표되었나요?",
    "LoRA의 핵심 원리는 무엇인가요?",
    "RAG는 어떤 문제를 해결하기 위한 기술인가요?",
    "DPO와 RLHF의 차이점은?",
    "QLoRA가 소비자급 GPU에서 학습 가능한 이유는?",
]

# Ground Truth (정답)
ground_truths = [
    "트랜스포머는 2017년 구글이 'Attention is All You Need' 논문에서 발표했습니다.",
    "LoRA는 전체 모델 파라미터를 동결하고 작은 저랭크 행렬만 학습하는 효율적 파인튜닝 방법입니다.",
    "RAG는 LLM의 환각(hallucination) 문제를 해결하기 위해 외부 지식을 검색하여 활용하는 기술입니다.",
    "DPO는 RLHF를 단순화한 방법으로, 보상 모델 없이 직접 최적화합니다.",
    "QLoRA는 4bit 양자화와 LoRA를 결합하여 메모리 사용량을 크게 줄이기 때문입니다.",
]

print(f"📋 평가 데이터셋: {len(eval_questions)}개 질문")
for i, (q, a) in enumerate(zip(eval_questions, ground_truths), 1):
    print(f"\n  {i}. ❓ {q}")
    print(f"     ✅ {a[:60]}...")

In [ ]:
# 🚀 RAG 시스템으로 답변 생성
import time

print("🚀 RAG 시스템으로 답변 생성 중...")
print("=" * 60)

rag_answers = []
rag_contexts = []

for i, question in enumerate(eval_questions, 1):
    print(f"\n  {i}/{len(eval_questions)}: {question[:40]}...")
    
    start = time.time()
    result = qa_chain.invoke({"query": question})
    elapsed = time.time() - start
    
    answer = result["result"]
    contexts = [doc.page_content for doc in result["source_documents"]]
    
    rag_answers.append(answer)
    rag_contexts.append(contexts)
    
    print(f"    💬 {answer[:80]}...")
    print(f"    ⏱️ {elapsed:.2f}초 | 📚 컨텍스트 {len(contexts)}개")

print(f"\n✅ 모든 답변 생성 완료!")

## 4️⃣ RAG 평가 지표 이해

RAGAS에서 제공하는 핵심 평가 지표를 이해합니다.

### 📊 RAGAS 핵심 지표

| 지표 | 평가 대상 | 설명 | 필요 데이터 |
|------|----------|------|------------|
| **Faithfulness** | 생성 | 답변이 컨텍스트에 근거하는지 | 질문, 답변, 컨텍스트 |
| **Answer Relevancy** | 생성 | 답변이 질문에 관련있는지 | 질문, 답변 |
| **Context Precision** | 검색 | 검색 결과의 정밀도 | 질문, 컨텍스트, 정답 |
| **Context Recall** | 검색 | 필요한 정보가 검색되었는지 | 질문, 컨텍스트, 정답 |

In [ ]:
# 📊 RAGAS 평가 지표 시각화
print("📊 RAGAS 평가 지표 상세 설명")
print("=" * 65)

metrics_info = [
    {
        "name": "Faithfulness (충실도)",
        "range": "0.0 ~ 1.0",
        "good": "> 0.8",
        "desc": "답변의 모든 주장이 검색된 컨텍스트에 근거하는지 측정",
        "formula": "근거있는 주장 수 / 전체 주장 수",
    },
    {
        "name": "Answer Relevancy (답변 관련성)",
        "range": "0.0 ~ 1.0",
        "good": "> 0.8",
        "desc": "답변이 원래 질문에 얼마나 관련있는지 측정",
        "formula": "답변에서 생성된 질문과 원래 질문의 유사도",
    },
    {
        "name": "Context Precision (컨텍스트 정밀도)",
        "range": "0.0 ~ 1.0",
        "good": "> 0.7",
        "desc": "검색된 문서 중 관련있는 문서의 비율",
        "formula": "관련 문서 수 / 검색된 전체 문서 수",
    },
    {
        "name": "Context Recall (컨텍스트 재현율)",
        "range": "0.0 ~ 1.0",
        "good": "> 0.7",
        "desc": "정답에 필요한 정보가 검색 결과에 포함되었는지",
        "formula": "검색된 관련 정보 / 정답에 필요한 전체 정보",
    },
]

for metric in metrics_info:
    print(f"\n🔹 {metric['name']}")
    print(f"   범위: {metric['range']} (좋음: {metric['good']})")
    print(f"   설명: {metric['desc']}")
    print(f"   계산: {metric['formula']}")

## 5️⃣ 임베딩 기반 평가 (코드 직접 구현)

LLM 없이 임베딩 벡터의 코사인 유사도만으로 평가하는 가장 간단한 방법입니다.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 📐 임베딩 기반 유사도 평가
eval_embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def cosine_similarity(a, b):
    """코사인 유사도 계산"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def evaluate_answer_relevancy(question, answer):
    """질문-답변 관련성을 임베딩 유사도로 평가"""
    q_emb = eval_embedder.encode(question)
    a_emb = eval_embedder.encode(answer)
    return float(cosine_similarity(q_emb, a_emb))

def evaluate_context_relevancy(question, contexts):
    """질문-컨텍스트 관련성을 임베딩 유사도로 평가"""
    q_emb = eval_embedder.encode(question)
    scores = []
    for ctx in contexts:
        c_emb = eval_embedder.encode(ctx)
        scores.append(float(cosine_similarity(q_emb, c_emb)))
    return np.mean(scores)

def evaluate_answer_correctness(answer, ground_truth):
    """답변-정답 유사도를 임베딩으로 평가"""
    a_emb = eval_embedder.encode(answer)
    g_emb = eval_embedder.encode(ground_truth)
    return float(cosine_similarity(a_emb, g_emb))

# 📊 전체 평가 실행
print("📊 임베딩 기반 수동 평가")
print("=" * 70)

all_scores = []

for i in range(len(eval_questions)):
    q = eval_questions[i]
    a = rag_answers[i]
    c = rag_contexts[i]
    g = ground_truths[i]
    
    answer_rel = evaluate_answer_relevancy(q, a)
    context_rel = evaluate_context_relevancy(q, c)
    correctness = evaluate_answer_correctness(a, g)
    
    scores = {
        "answer_relevancy": answer_rel,
        "context_relevancy": context_rel,
        "answer_correctness": correctness,
    }
    all_scores.append(scores)
    
    print(f"\n❓ Q{i+1}: {q[:40]}...")
    print(f"   답변 관련성:    {answer_rel:.4f}")
    print(f"   컨텍스트 관련성: {context_rel:.4f}")
    print(f"   답변 정확도:    {correctness:.4f}")

# 평균 점수
print(f"\n{'=' * 70}")
print(f"📊 평균 점수:")
for metric in ["answer_relevancy", "context_relevancy", "answer_correctness"]:
    avg = np.mean([s[metric] for s in all_scores])
    bar = "█" * int(avg * 20)
    print(f"  {metric:<25} {avg:.4f} {bar}")

## 6️⃣ LLM-as-a-Judge 구현

**LLM을 판사로** 사용하여 RAG 응답의 품질을 평가합니다.

### 🔑 LLM-as-a-Judge의 장점
- 🔹 인간 평가에 가까운 품질 판단
- 🔹 자동화 가능 (대규모 평가)
- 🔹 다양한 기준으로 평가 가능
- 🔹 이유(reasoning)도 함께 제공

In [ ]:
# 🧑‍⚖️ LLM-as-a-Judge 프롬프트 정의

JUDGE_PROMPT_FAITHFULNESS = """당신은 AI 응답의 품질을 평가하는 전문 평가자입니다.

아래의 컨텍스트와 답변을 보고 **충실도(Faithfulness)**를 평가해주세요.
충실도란 답변이 제공된 컨텍스트의 정보에 얼마나 충실한지를 의미합니다.

컨텍스트:
{context}

질문: {question}
답변: {answer}

평가 기준:
- 5점: 답변의 모든 내용이 컨텍스트에 근거함
- 4점: 대부분 근거하지만 약간의 추가 정보 포함
- 3점: 절반 정도 컨텍스트에 근거함
- 2점: 컨텍스트에 근거하지 않는 내용이 많음
- 1점: 컨텍스트와 거의 관련 없는 답변

다음 형식으로 답변하세요:
점수: [1-5]
이유: [평가 이유를 간결하게]"""

JUDGE_PROMPT_RELEVANCY = """당신은 AI 응답의 품질을 평가하는 전문 평가자입니다.

아래의 질문과 답변을 보고 **관련성(Relevancy)**을 평가해주세요.
관련성이란 답변이 질문에 얼마나 적절하게 대답하는지를 의미합니다.

질문: {question}
답변: {answer}

평가 기준:
- 5점: 질문에 완벽하게 대답함
- 4점: 대체로 잘 대답하지만 일부 누락
- 3점: 부분적으로만 대답함
- 2점: 질문과 약간만 관련 있는 답변
- 1점: 질문과 전혀 관련 없는 답변

다음 형식으로 답변하세요:
점수: [1-5]
이유: [평가 이유를 간결하게]"""

print("🧑‍⚖️ LLM-as-a-Judge 프롬프트 정의 완료")
print(f"  📝 충실도 평가 프롬프트: {len(JUDGE_PROMPT_FAITHFULNESS)}자")
print(f"  📝 관련성 평가 프롬프트: {len(JUDGE_PROMPT_RELEVANCY)}자")

In [ ]:
# 🧑‍⚖️ LLM-as-a-Judge 실행
import re

def extract_score(text):
    """LLM 응답에서 점수를 추출합니다."""
    match = re.search(r'점수[:\s]*([1-5])', text)
    if match:
        return int(match.group(1))
    return 3  # 기본값

def judge_response(question, answer, contexts, judge_type="relevancy"):
    """LLM을 사용하여 응답을 평가합니다."""
    if judge_type == "faithfulness":
        prompt = JUDGE_PROMPT_FAITHFULNESS.format(
            context="\n".join(contexts),
            question=question,
            answer=answer
        )
    else:
        prompt = JUDGE_PROMPT_RELEVANCY.format(
            question=question,
            answer=answer
        )
    
    result = llm.invoke(prompt)
    result_text = result if isinstance(result, str) else result.content
    score = extract_score(result_text)
    
    return score, result_text

# 평가 실행
print("🧑‍⚖️ LLM-as-a-Judge 평가 실행")
print("=" * 60)

judge_results = []

for i in range(len(eval_questions)):
    q = eval_questions[i]
    a = rag_answers[i]
    c = rag_contexts[i]
    
    print(f"\n❓ Q{i+1}: {q[:40]}...")
    
    # 충실도 평가
    try:
        faith_score, faith_reason = judge_response(q, a, c, "faithfulness")
        print(f"   📊 충실도: {faith_score}/5")
    except Exception as e:
        faith_score = 0
        faith_reason = str(e)
        print(f"   ❌ 충실도 평가 실패: {e}")
    
    # 관련성 평가
    try:
        rel_score, rel_reason = judge_response(q, a, c, "relevancy")
        print(f"   📊 관련성: {rel_score}/5")
    except Exception as e:
        rel_score = 0
        rel_reason = str(e)
        print(f"   ❌ 관련성 평가 실패: {e}")
    
    judge_results.append({
        "question": q,
        "faithfulness": faith_score,
        "relevancy": rel_score,
    })

In [ ]:
# 📊 LLM-as-a-Judge 결과 요약
print("📊 LLM-as-a-Judge 평가 결과 요약")
print("=" * 60)

valid_results = [r for r in judge_results if r["faithfulness"] > 0]

if valid_results:
    avg_faith = np.mean([r["faithfulness"] for r in valid_results])
    avg_rel = np.mean([r["relevancy"] for r in valid_results])
    
    print(f"\n  📊 평균 충실도 (Faithfulness): {avg_faith:.1f}/5.0")
    print(f"  📊 평균 관련성 (Relevancy):    {avg_rel:.1f}/5.0")
    
    print(f"\n  질문별 상세:")
    for r in judge_results:
        faith_bar = "⭐" * r["faithfulness"]
        rel_bar = "⭐" * r["relevancy"]
        print(f"    {r['question'][:35]:35s} 충실도: {faith_bar:<5s} 관련성: {rel_bar}")
else:
    print("⚠️ 유효한 평가 결과가 없습니다.")

## 7️⃣ RAGAS 자동 평가 실행

RAGAS 프레임워크를 사용하여 RAG 시스템을 자동으로 평가합니다.

In [ ]:
# 📊 RAGAS 평가 실행
from datasets import Dataset

# 평가 데이터셋 구성
eval_dataset = Dataset.from_dict({
    "question": eval_questions,
    "answer": rag_answers,
    "contexts": rag_contexts,
    "ground_truth": ground_truths,
})

print("📊 평가 데이터셋 구성")
print(f"  질문 수: {len(eval_dataset)}")
print(f"  컬럼: {eval_dataset.column_names}")
print(f"\n🔍 첫 번째 항목 미리보기:")
print(f"  질문: {eval_dataset[0]['question']}")
print(f"  답변: {eval_dataset[0]['answer'][:80]}...")
print(f"  컨텍스트 수: {len(eval_dataset[0]['contexts'])}")
print(f"  정답: {eval_dataset[0]['ground_truth'][:80]}...")

In [ ]:
# 🚀 RAGAS 평가 실행
try:
    from openai import OpenAI
    from ragas import evaluate
    from ragas.llms import llm_factory
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from langchain_community.embeddings import HuggingFaceEmbeddings

    print("🚀 RAGAS 평가 실행 중...")
    print("⏳ GPT-4o-mini로 평가하므로 1~2분 걸립니다...")

    # LLM: OpenAI GPT-4o-mini
    client = OpenAI()
    llm = llm_factory("gpt-4o-mini", client=client)

    # 임베딩: 로컬 HuggingFace (Langchain 래퍼 사용)
    emb = LangchainEmbeddingsWrapper(
        HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    )

    # 기본 4가지 지표로 평가 (metrics=None → 기본값 사용)
    result = evaluate(
        dataset=eval_dataset,
        llm=llm,
        embeddings=emb,
    )

    # 결과를 DataFrame으로 변환
    df = result.to_pandas()

    # 평가 지표 컬럼만 추출
    skip_cols = {"user_input", "retrieved_contexts", "response", "reference"}
    metric_cols = [c for c in df.columns if c not in skip_cols]

    print("\n📊 RAGAS 평가 결과 (질문별):")
    print("=" * 70)
    for i, row in df.iterrows():
        print(f"\n  Q{i+1}: {row['user_input'][:50]}...")
        for col in metric_cols:
            score = row[col]
            if isinstance(score, (int, float)):
                bar = "█" * int(score * 20)
                print(f"    {col:<25} {score:.4f} {bar}")

    print(f"\n{'=' * 70}")
    print("📊 평균 점수:")
    for col in metric_cols:
        avg = df[col].mean()
        bar = "█" * int(avg * 20)
        print(f"  {col:<25} {avg:.4f} {bar}")

except ImportError as e:
    print(f"⚠️ 패키지 오류: {e}")
    print("   pip install ragas openai")
except Exception as e:
    print(f"⚠️ 평가 중 오류 발생: {e}")
    print("   OpenAI API 키를 확인하세요.")


## 8️⃣ 평가 결과 분석 및 개선 방향

평가 결과를 바탕으로 RAG 시스템의 개선 방향을 도출합니다.

In [ ]:
# 💡 RAG 개선 방향 가이드
print("💡 평가 결과에 따른 RAG 개선 가이드")
print("=" * 60)

improvement_guide = [
    {
        "problem": "검색 관련성 낮음 (Context Precision ↓)",
        "solutions": [
            "청크 크기 조정 (더 작게 또는 더 크게)",
            "더 나은 임베딩 모델 사용 (multilingual-e5 등)",
            "HyDE(Hypothetical Document Embeddings) 적용",
            "검색 결과 수(k) 조정",
        ]
    },
    {
        "problem": "검색 재현율 낮음 (Context Recall ↓)",
        "solutions": [
            "문서 범위 확대",
            "청크 오버랩 증가",
            "쿼리 확장(Query Expansion) 적용",
            "Ensemble Retriever 사용 (BM25 + 시맨틱)",
        ]
    },
    {
        "problem": "충실도 낮음 (Faithfulness ↓)",
        "solutions": [
            "프롬프트에 '컨텍스트만 참조' 지시 강화",
            "더 큰 LLM 사용",
            "temperature 낮추기",
            "few-shot 예시 추가",
        ]
    },
    {
        "problem": "답변 관련성 낮음 (Answer Relevancy ↓)",
        "solutions": [
            "프롬프트 템플릿 개선",
            "검색 결과 Reranking 적용",
            "답변 후처리 (관련 없는 부분 제거)",
            "더 강력한 LLM 사용",
        ]
    },
]

for item in improvement_guide:
    print(f"\n🔴 문제: {item['problem']}")
    print(f"   해결 방법:")
    for i, sol in enumerate(item['solutions'], 1):
        print(f"   {i}. {sol}")

In [ ]:
# 📌 실습 정리
print("📌 오늘의 핵심 정리")
print("=" * 50)
print("  1️⃣ RAG 평가는 검색 품질과 생성 품질을 모두 측정")
print("  2️⃣ RAGAS는 Faithfulness, Relevancy 등 자동 평가 제공")
print("  3️⃣ 임베딩 유사도로 간단한 평가를 구현할 수 있음")
print("  4️⃣ LLM-as-a-Judge는 사람 평가에 가까운 품질 판단")
print("  5️⃣ 평가 결과를 바탕으로 체계적인 개선이 가능")

---

## 🎯 실습 과제

1️⃣ 자신의 RAG 시스템에 대해 평가 데이터셋 5개를 만들어 평가해보세요  
2️⃣ LLM-as-a-Judge 프롬프트를 수정하여 '완전성(Completeness)' 지표를 추가해보세요  
3️⃣ chunk_size를 변경한 후 평가 점수가 어떻게 달라지는지 비교해보세요  

---

## 📚 참고 자료
- [RAGAS 공식 문서](https://docs.ragas.io/)
- [RAGAS 논문](https://arxiv.org/abs/2309.15217)
- [LLM-as-a-Judge 논문](https://arxiv.org/abs/2306.05685)